In [ ]:
from os.path import join
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from os.path import dirname, abspath,join
import sys
ROOT_DIR = "../"
sys.path.append(ROOT_DIR)
from process_pipeline.core.labels import *

dataset_id = "dx_MSI_01_01_2022-07_07_2025"
INPUT_DIR,OUTPUT_DIR,INTERMEDIATE_DIR = get_dirs(ROOT_DIR, dataset_id)

In [ ]:
id_label = "id"
design_label = "SAP"
version_label = "Version"
batch_label = "WA"
step_label = "PaPos"
variable_label = "Variable"
process_label = "Process"
given_label = "Given"

In [ ]:
X_np = np.load(join(OUTPUT_DIR,"ds_"+dataset_id,"X.npy"))
Y_np = np.load(join(OUTPUT_DIR,"ds_"+dataset_id,"Y.npy"))
print(X_np.shape)
print(Y_np.shape)

In [ ]:
X_np[0,:,7]

In [ ]:
Y_np[0,:,-1]

In [ ]:
df = pd.read_parquet(join(OUTPUT_DIR,"df_input.parquet"))
missing_percent = df[trans_value_label].isna().mean()*100
missing_percent


In [ ]:
df

# Missing Values

In [ ]:
missing_percent_value = df[trans_value_label].isna().mean()*100
missing_percent_time = df[trans_date_label].isna().mean()*100
print(f"values missing: {missing_percent_value:.1f}%")
print(f"timestamps missing: {missing_percent_value:.1f}%")


In [ ]:
cat_col = ["position"]
hue_column = "process"
value_column = "value"  # Replace with your actual numerical/target column

missing_df = df.groupby(cat_col).agg(
    {
        value_column : lambda x: x.isna().mean() * 100,
        hue_column : "first"
    }).reset_index()

plt.figure(figsize=(10, 6))
sns.barplot(data=missing_df, x="position", y="value", hue="process")

In [ ]:

def filter_vars_max_missing(df: pd.DataFrame, threshold: float)->pd.DataFrame:
    """
    Filters process dataframe variables that contains a percentage of missing
    values > threshold

    Args:
        df (pd.DataFrame): non-filtered dataframe
        threshold (float): threshold

    Returns:
        pd.DataFrame: filtered dataframe
    """
    
    # calculate total percentage of missing values before filtering
    missing_percent_before = df[trans_value_label].isna().mean()*100

    # calculate % missing for each variable 
    missing_df = df.groupby(trans_variable_label).agg(
        {
            trans_value_label : lambda x: x.isna().mean() * 100,
        }).reset_index()
    
    # get variables where % missing < threshold
    filter_vars = missing_df[missing_df[trans_value_label] <= threshold][trans_variable_label]
    
    # filter variables
    df_filtered = df[df[trans_variable_label].isin(filter_vars)]
    
    # calculate total percentage of missing values after filtering
    missing_percent_after = df_filtered[trans_value_label].isna().mean()*100

    return df_filtered


In [ ]:
missing_percent = df_filtered[trans_value_label].isna().mean()*100
missing_percent

 # DataFrame preparation
 ## Check weird Plasma double steps

In [ ]:
sel_design = 453828
sel_version = "B"

df_sel = df_lev.set_index([input_design_label, input_version_label]).loc[sel_design].loc[sel_version].reset_index()

In [ ]:
# check weird plasma step
steps_check = [251,271,461,481] #steps where plasma overlaps with other processes
for step in steps_check:
    df_sel_inspect = df_sel.set_index(input_step_label).loc[step][input_value_label]
    print(f"Step: {step}")
    print("Is Nan: ", df_sel_inspect.isna().sum())
    print("Not Nan: ", df_sel_inspect.notna().sum())
    print("Not nan but zero: ", df_sel_inspect[(df_sel_inspect.notna()) & (df_sel_inspect==0)].count())
    print("Not nan not zero: ", df_sel_inspect[(df_sel_inspect.notna()) & (df_sel_inspect!=0)].count())
    
#TODO: which variable is not zero?

In [ ]:
df_plasma = pd.read_csv("../data/input/plasma.csv",sep = ";",skiprows=11)
df_plasma_sel = df_plasma.set_index(["ArtikelNr","ArtikelVersion"]).loc[sel_design].loc[sel_version]
df_plasma_sel.head()

In [ ]:
# concatenate Dyconex feedback with scripts output
# df_sel_steps_pro = df_sel[[input_step_label,input_process_label]].drop_duplicates()
# #df_lev_steps_pro.to_excel("../data/output/steps_process_list.xlsx")
# df_Dyconex_survey = pd.read_excel("../data/input/Dyconex_survey_436425.xlsx", usecols=[0, 1, 2])
# #df_lev_steps_pro.to_excel("../data/output/steps_process_list.xlsx")
# pd.concat([df_Dyconex_survey.set_index("Pos"),df_sel_steps_pro.set_index(input_step_label)],axis=1).to_excel("../data/output/steps_process_list_survey.xlsx")

## Select steps

In [ ]:
df_steps_sel = pd.read_excel("../data/intermediate/steps_selected.xlsx")
steps = np.array(df_steps_sel[df_steps_sel['Select']]["Step"])
filtered_data = df_sel[df_sel[input_step_label].isin(steps)]
filtered_data.to_csv("../data/intermediate/x_prochain_lev_sel.csv")

# Data info plots

In [ ]:
f, ax = plt.subplots(figsize=(7, 5))

sns.despine(f)

df_plot = df_pc.copy()
df_plot["Time"] = pd.to_datetime(df_plot["Time"]).round("30D")
df_plot = df_plot.sort_values("Time")

sns.histplot(data=df_plot, x="Time", hue="SAP", multiple="stack",
             palette="light:m_r",edgecolor=".3",linewidth=.5,bins=20,stat="percent", ax = ax)

plt.xticks(rotation=45)
plt.show()

In [ ]:
processes= df_pc["Process"].unique()
designs = df_pc[design_label].unique()
processes

In [ ]:
df_plot = df_lev.copy()
df_plot["SAP"] = df_plot["SAP"].apply(lambda x: str(x))
df_plot["SAP_Version"] = df_plot["SAP"]+"_"+df_plot["Version"]

f, ax = plt.subplots(figsize=(7, 5))
sns.despine(f)

sns.histplot(
    data = df_plot,
    x="SAP_Version", 
    hue="Process", 
    hue_order=processes,
    multiple="stack",
    palette="Spectral",
    edgecolor=".3",
    linewidth=.5,
    stat="percent")

plt.xticks(rotation=45)
plt.show()

# Data Leveling
## Before Leveling

In [ ]:
multi = designs
sub_label = design_label


f, ax = plt.subplots(1,len(multi),figsize=(7*len(multi), 5))
sns.despine(f)

for i, sub_class in enumerate(multi):
        
    sns.histplot(
        data = df_pc.set_index(design_label).loc[sub_class],
        x=id_label, hue=version_label, multiple="stack",palette="Spectral",edgecolor=".3",linewidth=0,ax=ax[i])
    
    ax[i].axes.get_xaxis().set_ticks([])
    ax[i].set_title(f"Design {sub_class}")

## After Leveling

In [ ]:
f, ax = plt.subplots(1,len(multi),figsize=(7*len(multi), 5))
sns.despine(f)

for i, sub_class in enumerate(multi):
        
    sns.histplot(
        data = df_lev.set_index(design_label).loc[sub_class],
        x=id_label, hue=version_label, multiple="stack",palette="Spectral",edgecolor=".3",linewidth=0,ax=ax[i])
    
    ax[i].axes.get_xaxis().set_ticks([])
    ax[i].set_title(f"Design {sub_class}")

In [ ]:
processes = df_lev[input_process_label].unique()
processes

In [ ]:
f, ax = plt.subplots(1,len(multi),figsize=(7*len(multi), 5))
sns.despine(f)



for i, sub_class in enumerate(multi):
    
    data = df_lev.set_index(design_label).loc[sub_class]
    
    sns.histplot(
        data = data,
        x=input_id_label, hue=input_process_label, hue_order=processes, 
        multiple="stack",palette="Spectral",edgecolor=".3",linewidth=0,ax=ax[i])
    
    ax[i].axes.get_xaxis().set_ticks([])
    ax[i].set_title(f"Design {sub_class}")

## Lot's of voids

In [ ]:
f, ax = plt.subplots(1,len(multi),figsize=(7*len(multi), 5))
sns.despine(f)

for i, sub_class in enumerate(multi):
        
    sns.histplot(
        data = df_lev.set_index(design_label).loc[sub_class],
        x=id_label, hue=given_label, multiple="stack",palette="Spectral",edgecolor=".3",linewidth=0,ax=ax[i])
    
    ax[i].axes.get_xaxis().set_ticks([])
    ax[i].set_title(f"Design {sub_class}")

## Can we take away some variables?
A part from some exceptions where the variables are completely missing, most of them are in some data and other don't

In [ ]:
multi = processes
sub_label = process_label

f, ax = plt.subplots(1,len(multi),figsize=(7*len(multi), 5))
sns.despine(f)

for i, sub_class in enumerate(multi):
        
    sns.histplot(
        data = df_lev.set_index(sub_label).loc[sub_class],
        x=variable_label, hue=given_label, multiple="stack",palette="Spectral",edgecolor=".3",linewidth=0,ax=ax[i])
    
    ax[i].axes.get_xaxis().set_ticks([])
    ax[i].set_title(f"Subclass {sub_class}")